In [ ]:

import pandas as pd
from pathlib import Path
import calendar

data_dir = Path("../data/raw")

files = {
    8: "data_barrages_tensift_aout_2026.xlsx",
    9: "data_barrages_tensift_sept_2026.xlsx",
    10: "data_barrages_tensift_oct_2026.xlsx"
}

def clean_dams(file_path, month) -> pd.DataFrame:

    df = pd.read_excel(file_path, header=None)
    file_dams_data = []

    # Find the row containing "JOUR"
    header_rows = df.index[
        df.iloc[:, 0].astype(str).str.strip() == "JOUR"
    ]
    if len(header_rows) != 1 or header_rows[0] < 1:
        raise ValueError(f"{file_path}: expected one JOUR header after a capacity row")
    if df.shape[1] < 11:
        raise ValueError(f"{file_path}: expected five reserve/fill-rate column pairs")
    header_row = header_rows[0]

    capacity_row = header_row - 1
    start_row = header_row + 1

    # Extract days and select valid day rows
    jours = pd.to_numeric(
        df.iloc[start_row:, 0], errors="coerce"
    )

    max_day = calendar.monthrange(2026, month)[1]
    valid = jours.between(1, max_day) & jours.mod(1).eq(0)

    for i in range(1, 11, 2):

        dam_name = str(df.iloc[0, i]).strip()

        capacite = pd.to_numeric(
            df.iloc[capacity_row, i], errors="coerce"
        )

        reserve = pd.to_numeric(
            df.iloc[start_row:, i], errors="coerce"
        )

        taux = pd.to_numeric(
            df.iloc[start_row:, i + 1], errors="coerce"
        )

        clean_data = pd.DataFrame({
            "jour": jours[valid].reset_index(drop=True).astype(int),
            "dam": dam_name,
            "capacity_mm3": capacite,
            "reserve_mm3": reserve[valid].reset_index(drop=True),
            "fill_rate": taux[valid].reset_index(drop=True)
        })

        clean_data["date"] = pd.to_datetime({
            "year": 2026,
            "month": month,
            "day": clean_data["jour"]
        })

        clean_data = clean_data[
            ["date", "dam", "capacity_mm3",
             "reserve_mm3", "fill_rate"]
        ]

        file_dams_data.append(clean_data)

    return pd.concat(file_dams_data, ignore_index=True)







In [23]:
monthly_data = []

for month, filename in files.items():
    file_path = data_dir / filename

    monthly_df = clean_dams(file_path, month)
    monthly_data.append(monthly_df)

all_dams = pd.concat(monthly_data, ignore_index=True)


all_dams = all_dams.sort_values(
    ["dam", "date"]
).reset_index(drop=True)



In [ ]:
print("Shape:", all_dams.shape)
print("\nFirst rows:")
print(all_dams.head())
print("\nMissing values:")
print(all_dams.isna().sum())
print("\nUnique dams:", all_dams["dam"].nunique())
print("\nDuplicates:", all_dams.duplicated(
    subset=["date", "dam"]
).sum())


output = Path("../data/processed/tensift_cleaned.csv")
output.parent.mkdir(parents=True, exist_ok=True)

all_dams.to_csv(output, index=False)

In [24]:
august = clean_dams(
    "../data/raw/data_barrages_tensift_aout_2026.xlsx",
    8
)
september = clean_dams(
    "../data/raw/data_barrages_tensift_sept_2026.xlsx",
    9
)
october = clean_dams(
    "../data/raw/data_barrages_tensift_oct_2026.xlsx",
    10
)

print(september)
print(august)
print(october)

          date                dam  capacity_mm3  reserve_mm3  fill_rate
0   2026-09-01  Yacoub El Mansour         57.45       49.444  86.064404
1   2026-09-02  Yacoub El Mansour         57.45       49.444  86.064404
2   2026-09-03  Yacoub El Mansour         57.45       49.444  86.064404
3   2026-09-04  Yacoub El Mansour         57.45       49.444  86.064404
4   2026-09-05  Yacoub El Mansour         57.45       49.468  86.106179
..         ...                ...           ...          ...        ...
145 2026-09-26   Bge My abdrhmane         58.99       48.474  82.173250
146 2026-09-27   Bge My abdrhmane         58.99       48.391  82.032548
147 2026-09-28   Bge My abdrhmane         58.99       48.352  81.966435
148 2026-09-29   Bge My abdrhmane         58.99       48.318  81.908798
149 2026-09-30   Bge My abdrhmane         58.99       48.236  81.769791

[150 rows x 5 columns]
          date                dam  capacity_mm3  reserve_mm3  fill_rate
0   2026-08-01  Yacoub El Mansour       